<a href="https://colab.research.google.com/github/saffarizadeh/INSY5378/blob/main/Assignments/Assignment_6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<img src="https://kambizsaffari.com/Logo/College_of_Business.cmyk-hz-lg.png" width="500px"/>

# *INSY 5378*

# **Assignment: Training a Mini-GPT from Scratch**

Instructor: Dr. Kambiz Saffari

---

**What this assignment is**

This is a hands-on walkthrough of **Chapter 16: Text Generation**. You are not expected to read the chapter cover to cover before starting. Instead, **use the chapter as your tutorial while you work through this notebook.** Each part of this assignment points you to the exact section or listing number in the chapter that explains what you need to do. Open the chapter alongside this notebook, read one section, implement it here, run it, and move on to the next.

The chapter is available online at: https://deeplearningwithpython.io/chapters/chapter16_text-generation/

**How to work through it**
- Treat the chapter as a guided tour. Every piece of code you need is already explained and shown in listings. Your job is to follow along, type it out yourself, run it, and answer the short reflection questions so you actually understand what you built.
- Write your answers directly in the provided cells. You may add extra cells to experiment.
- The reflection comments are where you demonstrate understanding. Do not skip them. Copy-pasting code without thinking about it will not earn full credit.

**Why this assignment matters**

This is the final assignment of the course, and it has two goals.

The first goal is technical. Up to this point, you have probably used language models mostly as black boxes. Here you will build one yourself. You will assemble a decoder-only Transformer, pretrain it on roughly a billion tokens of real web text, and generate new text with several different sampling strategies. By the end, you will have touched every major ingredient that powers modern LLMs like GPT and Gemma: subword tokenization, causal self-attention, positional embeddings, weight tying, learning rate warmup, mixed precision, and temperature based sampling. The model you train will be tiny compared to production LLMs, but the recipe is exactly the same one used at industrial scale.

The second goal is more important and will matter long after this course ends. Deep learning moves fast. The specific models, libraries, and **best practices we covered in class will look different a year from now, and very different five years from now. No single course can keep you current. What *will* keep you current is the ability to sit down with a high-quality online resource (a book chapter, a paper, an official tutorial, a well-written blog post) and work through it carefully enough to actually understand and reproduce what it teaches. That is a skill, and like any skill it needs practice.** This assignment is that practice. You are being asked to take a chapter from a book, a chapter you have not read yet, open it alongside your notebook, and follow it end to end until you have a working mini-GPT. Treat this as a dress rehearsal for how you will learn every new technique that comes out after you leave this class.

**Compute requirements**

This is the most compute intensive assignment of the course. You should run it on **Google Colab with a GPU runtime** (Runtime > Change runtime type > T4 GPU or better). Training the mini-GPT will take a few hours on a free T4 GPU. If you run out of free GPU credits, remember that as a UTA student you can activate **Colab Pro through your UTA student email** for access to faster GPUs and longer runtimes. Start early. Do not leave this assignment to the last day.

Before committing to the full training run, get the code working end to end. The chapter notes that you can temporarily reduce `steps_per_epoch` and `num_epochs` to small values while you debug, then restore them for your real run.

**Disclaimer:** The text generated by your mini-GPT is purely the output of a small model trained on a subset of web data for educational purposes. It may be incoherent, repetitive, or contain artifacts of its training data. Do not treat any generated output as factual.

## Setup

Run the cells below to install dependencies and configure the backend. Make sure you are on a GPU runtime before you continue.

In [ ]:
import os
os.environ["KERAS_BACKEND"] = "tensorflow"

In [ ]:
!pip install keras keras-hub --upgrade -q

## Question 1: Data, Tokenization, and Input Pipeline

**Work alongside:** The sections **"Training a mini-GPT"** (intro and data download) in Chapter 16. Keep the chapter open. Everything you need in this question is shown in Listings 16.1, 16.2, and 16.3. Follow along and type the code into the cell below.

---

**Part A: Download the Pretraining Data**

1. Follow Listing 16.1 to download the mini C4 dataset using `keras.utils.get_file()`. Store the extracted directory in a variable called `extract_dir`.
2. Use `pathlib` to list the shard files inside `extract_dir`. Print how many shard files you found.
3. Open the first shard and print the first 200 characters of the first document (remember to replace the escaped `\n` with real newlines as the chapter does).
4. In a comment, briefly describe what kind of text appears in this sample. Is it clean? Is it noisy? What do you notice about it as pretraining data?

**Part B: Tokenization with SentencePiece**

1. Download the premade SentencePiece vocabulary file and instantiate a `keras_hub.tokenizers.SentencePieceTokenizer` exactly as shown in Listing 16.2.
2. Tokenize the sentence `"Deep learning is fun."` and print the resulting integer IDs.
3. Call `detokenize()` on those same IDs and confirm that you recover the original string.
4. Print the vocabulary size of your tokenizer.
5. In a comment, explain in your own words why subword tokenization (like SentencePiece / BPE) is preferred over simple word level tokenization for an LLM. Reference the chapter discussion.

**Part C: Building the `tf.data` Pipeline**

1. Reproduce Listing 16.3 to build the training pipeline. Use `batch_size=64` and `sequence_length=256`. Append the `<|endoftext|>` token to each document and window the data into sequences of length `sequence_length + 1`. Split each window into `(x[:-1], x[1:])` so the label at each position is the next token.
2. Take one batch from the dataset and print the shape of the input `x` and the target `y`. In a comment, explain what each dimension represents and why `y` is just `x` shifted by one position.
3. Following the chapter, split off the first **500 batches** as a repeating validation set and use the rest as a repeating training set. Store them as `val_ds` and `train_ds`. Set `num_train_batches` and `num_val_batches` as in the chapter (`num_batches = 58746`).

In [ ]:
# Write your answer to Question 1 here


## Question 2: Building, Training, and Sampling from a Mini-GPT

**Work alongside:** The sections **"Building the model"**, **"Pretraining the model"**, **"Generative decoding"**, and **"Sampling strategies"** in Chapter 16. The listings in this question (16.4 through 16.10) give you the exact code. Read one listing, type it into the cell below, run it, then move to the next.

---

**Part A: Build the Decoder-Only Transformer**

1. Implement the `TransformerDecoder` layer from Listing 16.4. This is a decoder block with causal self attention, a feedforward network, residual connections, layer normalization, and dropout. There is **no cross attention**.
2. Implement the `PositionalEmbedding` layer from Listing 16.5. It should support a `reverse=True` mode that projects hidden states back to vocabulary logits using the transpose of the token embedding matrix.
3. In a comment, explain in your own words the **weight tying** trick used here. Why does sharing the same matrix for the input token embedding and the output projection save memory, and why does the chapter argue it is a reasonable thing to do?
4. Enable mixed precision with `keras.config.set_dtype_policy("mixed_float16")` and build the functional mini-GPT model exactly as shown in Listing 16.6, with `hidden_dim=512`, `intermediate_dim=2056`, `num_heads=8`, and `num_layers=8`. Call `mini_gpt.summary()` and report the total parameter count.
5. In a comment, explain why the `TransformerDecoder` uses `use_causal_mask=True` in its self attention call. What would go wrong during pretraining if this mask were removed?

**Part B: Pretraining with Warmup**

1. Implement the `WarmupSchedule` learning rate schedule from Listing 16.7. Plot the learning rate over the first 5,000 steps to confirm it looks like the warmup curve in Figure 16.2.
2. In a comment, explain why a warmup schedule is important when training a deep Transformer from scratch. What problem does it help prevent?
3. Compile the model with `Adam(schedule)` and `SparseCategoricalCrossentropy(from_logits=True)`, and include `"accuracy"` as a metric.
4. Train the model following Listing 16.8 with `num_epochs=8` and `steps_per_epoch = num_train_batches // num_epochs`. **Expect this to take a few hours on a Colab T4.** Do not interrupt training once it is running well.
5. Report the final training and validation accuracy. In a comment, state whether your final validation accuracy is close to the chapter's reported value of about 36%. If yours is different, speculate on why.

**Part C: Generating Text with Sampling Strategies**

1. Implement the `compiled_generate()` function from Listing 16.10, which pads the input to a fixed length so that `predict()` does not recompile on every call. Use it to generate text for the prompt `"A piece of advice"` with `max_length=64`. Print the output.
2. Refactor `compiled_generate()` to accept a `sample_fn` argument as shown in the chapter. Implement the three sampling strategies from Chapter 16:
   - `greedy_search(preds)`
   - `random_sample(preds, temperature=1.0)`
   - `top_k(preds, k=5, temperature=1.0)`
3. Using the **same prompt** `"A piece of advice"`, generate one sample with each of the following settings and print all of them clearly labeled:
   - Greedy search
   - Random sampling with `temperature=1.0`
   - Random sampling with `temperature=0.2`
   - Random sampling with `temperature=2.0`
   - Top-k with `k=5`, `temperature=1.0`
   - Top-k with `k=20`, `temperature=1.0`
4. In a comment, compare the outputs. Specifically address:
   - What happens at very low temperature? What happens at very high temperature?
   - How does top-k sampling differ from temperature sampling in what it does to the probability distribution?
   - Which setting produced the output you thought was the most readable, and why?

**Part D: Reflection**

Answer each of the following in a short paragraph (a few sentences each). You can write your answers as comments in a code cell or as a markdown cell.

1. Your mini-GPT has about 41 million parameters and saw roughly 1 billion tokens during training. GPT-3 has about 175 billion parameters and was trained on roughly 500 billion tokens. Based on the generations you produced and the chapter discussion, describe two specific weaknesses of your mini-GPT that you would expect scaling to fix, and one weakness you would expect scaling **not** to fix.
2. The chapter emphasizes that even a large, well trained LLM is "effectively a fancy autocomplete for the internet." Using your own results as evidence, explain what this phrase means and why techniques like instruction fine tuning and RLHF are needed on top of raw pretraining to produce something like ChatGPT.
3. Training your mini-GPT took a few hours on a single small GPU. The chapter notes that Llama 2 required an estimated 1.3 million kilowatt hours of electricity to train. Briefly reflect on the implications of this for who gets to build frontier LLMs and for the environmental footprint of the field.

In [ ]:
# Write your answer to Question 2 here


---
**Submission Reminder**
- Make sure your notebook runs from top to bottom without errors.
- Clearly label all answers.
- Include all required comments and reflections. They are part of your grade.
- Because training takes several hours, **do not clear your outputs before submitting**. Your saved training logs and generated samples are part of what will be graded.